# NLP Lecture 2: Statistical Language Model -- Tokenization, N-gram Language Models, Smoothing, and BPE

This notebook covers:

1. Tokenization (simple, regex-based)
2. Unigram and bigram language models
3. Smoothing and simple backoff
4. A toy spam vs ham classifier using language models
5. Subword tokenization via **Byte Pair Encoding (BPE)** on a larger open corpus (WikiText-2 or a local text file)

All code is self-contained and should run in Colab (internet needed for WikiText-2 download unless you provide a local corpus file).

In [6]:
import re
import math
import os
import sys
import subprocess
from collections import Counter, defaultdict
from typing import List, Tuple

print("Environment ready.")

Environment ready.


## 1. Tokenization

We start with a very simple tokenizer:

- Lowercasing
- Splitting on words and punctuation
- No fancy linguistic handling (no stemming, no lemmatization)

This is **deliberately simple** and is just good enough for our n-gram language models later.

In [7]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text: str) -> List[str]:
    """Simple regex-based tokenizer:
    - lowercasing
    - splits off punctuation as separate tokens
    """
    text = text.lower()
    return TOKEN_PATTERN.findall(text)

def tokenize_keep_case(text: str) -> List[str]:
    """Simple regex-based tokenizer:
    - lowercasing
    - splits off punctuation as separate tokens
    """
    # text = text.lower()
    return TOKEN_PATTERN.findall(text)

# Quick sanity check
examples = [
    "I didn't say 'unbelievable'!!!",
    "The cats in the cradle and the silver spoon.",
    "Email: I am foreign royalty fleeing my home country..."
]

for s in examples:
    print("tokenize")
    print(s)
    print(tokenize(s))
    print("-" * 40)
    print("tokenize_keep_case")
    print(s)
    print(tokenize_keep_case(s))
    print("-" * 40)

tokenize
I didn't say 'unbelievable'!!!
['i', 'didn', "'", 't', 'say', "'", 'unbelievable', "'", '!', '!', '!']
----------------------------------------
tokenize_keep_case
I didn't say 'unbelievable'!!!
['I', 'didn', "'", 't', 'say', "'", 'unbelievable', "'", '!', '!', '!']
----------------------------------------
tokenize
The cats in the cradle and the silver spoon.
['the', 'cats', 'in', 'the', 'cradle', 'and', 'the', 'silver', 'spoon', '.']
----------------------------------------
tokenize_keep_case
The cats in the cradle and the silver spoon.
['The', 'cats', 'in', 'the', 'cradle', 'and', 'the', 'silver', 'spoon', '.']
----------------------------------------
tokenize
Email: I am foreign royalty fleeing my home country...
['email', ':', 'i', 'am', 'foreign', 'royalty', 'fleeing', 'my', 'home', 'country', '.', '.', '.']
----------------------------------------
tokenize_keep_case
Email: I am foreign royalty fleeing my home country...
['Email', ':', 'I', 'am', 'foreign', 'royalty', 'fle

## 2. N-gram Language Models

We will build **unigram** and **bigram** language models on a small toy corpus.

We add special tokens:

- `<s>`  : start of sentence
- `</s>` : end of sentence

Then we compute:

- Unigram counts and probabilities $ p(w) $
- Bigram counts and probabilities $ p(w_i \mid w_{i-1}) $

First we define some helpers to prepare the corpus and build the count tables.

In [8]:
def prepare_corpus(raw_sentences: List[str]) -> List[List[str]]:
    """Convert list of raw sentences into list of tokenized sentences
    with <s> and </s> markers."""
    corpus = []
    for s in raw_sentences:
        toks = tokenize(s)
        if not toks:
            continue
        corpus.append(["<s>"] + toks + ["</s>"])
    return corpus

def prepare_corpus_keep_case(raw_sentences: List[str]) -> List[List[str]]:
    """Convert list of raw sentences into list of tokenized sentences
    with <s> and </s> markers."""
    corpus = []
    for s in raw_sentences:
        toks = tokenize_keep_case(s)
        if not toks:
            continue
        corpus.append(["<s>"] + toks + ["</s>"])
    return corpus

def build_unigram_counts(corpus: List[List[str]]) -> Counter:
    counts = Counter()
    for sent in corpus:
        counts.update(sent) # 会自动统计每个word出现的次数
    return counts

def build_bigram_counts(corpus: List[List[str]]) -> Counter:
    counts = Counter()
    for sent in corpus:
        for w_prev, w_cur in zip(sent[:-1], sent[1:]): # 把句子前n-1个和后n-1个进行配对
            counts[(w_prev, w_cur)] += 1
    return counts

def build_trigram_counts(corpus: List[List[str]]) -> Counter:
    counts = Counter()
    for sent in corpus:
        for w1, w2, w3 in zip(sent[:-2], sent[1:-1], sent[2:]):
            counts[(w1, w2, w3)] += 1
    return counts

# A tiny toy corpus
toy_sentences = [
    "Its water is so transparent.",
    "Its water is so translucent.",
    "The students opened their laptops.",
    "The students opened their books.",
]

corpus = prepare_corpus(toy_sentences)
corpus_keep_case = prepare_corpus_keep_case(toy_sentences)
for sent in corpus:
    print(sent)

uni_counts = build_unigram_counts(corpus)
bi_counts = build_bigram_counts(corpus)
tri_counts = build_trigram_counts(corpus)

print("\nUnigram counts (top 10):")
for w, c in uni_counts.most_common(10):
    print(f"{w:12s} -> {c}")

print("\nBigram counts (sample):")
for (w_prev, w_cur), c in list(bi_counts.items())[:10]:
    print(f"({w_prev!r}, {w_cur!r}) -> {c}")

print("\nTrigram counts (sample):")
for (w1, w2, w3), c in list(tri_counts.items())[:10]:
    print(f"({w1!r}, {w2!r}, {w3!r}) -> {c}")

['<s>', 'its', 'water', 'is', 'so', 'transparent', '.', '</s>']
['<s>', 'its', 'water', 'is', 'so', 'translucent', '.', '</s>']
['<s>', 'the', 'students', 'opened', 'their', 'laptops', '.', '</s>']
['<s>', 'the', 'students', 'opened', 'their', 'books', '.', '</s>']

Unigram counts (top 10):
<s>          -> 4
.            -> 4
</s>         -> 4
its          -> 2
water        -> 2
is           -> 2
so           -> 2
the          -> 2
students     -> 2
opened       -> 2

Bigram counts (sample):
('<s>', 'its') -> 2
('its', 'water') -> 2
('water', 'is') -> 2
('is', 'so') -> 2
('so', 'transparent') -> 1
('transparent', '.') -> 1
('.', '</s>') -> 4
('so', 'translucent') -> 1
('translucent', '.') -> 1
('<s>', 'the') -> 2

Trigram counts (sample):
('<s>', 'its', 'water') -> 2
('its', 'water', 'is') -> 2
('water', 'is', 'so') -> 2
('is', 'so', 'transparent') -> 1
('so', 'transparent', '.') -> 1
('transparent', '.', '</s>') -> 1
('is', 'so', 'translucent') -> 1
('so', 'translucent', '.') -> 1
('t

### 2.1 Maximum-Likelihood Estimates (Unsmoothed)

We define a small `NGramLM` class supporting unigram and bigram models with **unsmoothed** probabilities:

- Unigram:  
  $  p(w) = \frac{\text{count}(w)}{N} $

- Bigram:  
  $ p(w_i \mid w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})} $

- Trigram:
  $ p(w_i \mid w_{i-1}, w_{i-2}) = \frac{\text{count}(w_{i-2}, w_{i-1}, w_i)}{\text{count}(w_{i-1}, w_{i-2})} $

Zeros for unseen events are expected here; we fix that later with smoothing.

In [9]:
class NGramLM:
    def __init__(self, corpus: List[List[str]], n: int = 2):
        assert n in (1, 2, 3)
        self.n = n
        self.unigrams = build_unigram_counts(corpus)
        self.total_tokens = sum(self.unigrams.values())
        self.vocab = set(self.unigrams.keys())
        self.V = len(self.vocab)

        if n == 2:
            self.bigrams = build_bigram_counts(corpus)
            self.context_counts = Counter()
            for (w_prev, w_cur), c in self.bigrams.items():
                self.context_counts[w_prev] += c
        if n == 3:
            self.trigrams = build_trigram_counts(corpus)
            self.context_counts = Counter()
            for (w1, w2, w3), c in self.trigrams.items():
                self.context_counts[(w1, w2)] += c

    def p_unigram(self, w: str) -> float:
        return self.unigrams[w] / self.total_tokens if self.unigrams[w] > 0 else 0.0

    def p_bigram_mle(self, w_prev: str, w: str) -> float:
        if self.n == 1:
            return self.p_unigram(w)
        num = self.bigrams[(w_prev, w)]
        den = self.context_counts[w_prev]
        if den == 0:
            return 0.0
        return num / den

    def p_trigram_mle(self, w1: str, w2: str, w3: str) -> float:
        if self.n == 1:
            return self.p_unigram(w3)
        if self.n == 2:
            return self.p_bigram_mle(w2, w3)
        num = self.trigrams[(w1, w2, w3)]
        den = self.context_counts[(w1, w2)]
        if den == 0:
            return 0.0
        return num / den

lm_unigram = NGramLM(corpus, n=1)
lm_bigram = NGramLM(corpus, n=2)
lm_trigram = NGramLM(corpus, n=3)

print("p_MLE('transparent' | 'so') =", lm_bigram.p_bigram_mle("so", "transparent"))
print("p_MLE('translucent' | 'so') =", lm_bigram.p_bigram_mle("so", "translucent"))
print("p_MLE('students' | 'the')   =", lm_bigram.p_bigram_mle("the", "students"))



p_MLE('transparent' | 'so') = 0.5
p_MLE('translucent' | 'so') = 0.5
p_MLE('students' | 'the')   = 1.0


### 2.2 Sentence Log-Probability and Perplexity (Unsmoothed)

For a bigram LM, the log-probability of a sentence $  w_1, \ldots, w_n $ is:

$ \log P(w_1^n) = \sum_i \log p(w_i \mid w_{i-1}) $

Perplexity on a corpus of \( N \) tokens is:

$ \text{PP} = \exp\left( - \frac{1}{N} \sum_i \log p(w_i \mid w_{i-1}) \right). $

We implement these without smoothing (so PP may be infinite if a bigram is unseen).

In [10]:
def sentence_logprob(lm: NGramLM, sent_tokens: List[str]) -> float:
    """Compute log-probability of a sentence under a unigram or bigram LM.
    sent_tokens should already include <s> and </s>."""
    logp = 0.0

    if lm.n == 1:
        for w in sent_tokens:
            p = lm.p_unigram(w)
            if p == 0:
                return float("-inf")
            logp += math.log(p)
    elif lm.n == 2:  # bigram
        for w_prev, w_cur in zip(sent_tokens[:-1], sent_tokens[1:]):
            p = lm.p_bigram_mle(w_prev, w_cur)
            if p == 0:
                return float("-inf")
            logp += math.log(p)
    else: # trigram
        for w1, w2, w3 in zip(sent[:-2], sent[1:-1], sent[2:]):
            p = lm.p_trigram_mle(w1, w2, w3)
            if p == 0:
                return float("-inf")
            logp += math.log(p)
    return logp


def corpus_perplexity(lm: NGramLM, corpus: List[List[str]]) -> float:
    total_logp = 0.0
    total_tokens = 0

    for sent in corpus:
        lp = sentence_logprob(lm, sent)
        total_logp += lp
        total_tokens += len(sent) - 1

    return math.exp(- total_logp / total_tokens)


print("Perplexity (unsmoothed) on training corpus:")
print("Unigram:", corpus_perplexity(lm_unigram, corpus))
print("Bigram:", corpus_perplexity(lm_bigram, corpus))
print("Trigram:", corpus_perplexity(lm_trigram, corpus))

Perplexity (unsmoothed) on training corpus:
Unigram: 19.504218467271595
Bigram: 1.2190136542044754
Trigram: 1.1040895136738123


## 3. Smoothing and Simple Backoff

Unsmoothed MLE gives probability 0 for unseen bigrams, which is a problem.

We fix this with **add-α (Laplace) smoothing** and a simple **backoff** strategy:

- Bigram add-α:

$ p(w \mid h) = \frac{\text{count}(h,w) + \alpha}{\text{count}(h) + \alpha V} $
- Unigram add-α similar.

- Backoff: if a bigram has zero count, we fall back to the smoothed unigram probability for that word.

In [11]:
class SmoothedBigramLM(NGramLM):
    def __init__(self, corpus: List[List[str]], alpha: float = 0.5):
        super().__init__(corpus, n=2)
        self.alpha = alpha

    def p_bigram_add_alpha(self, w_prev: str, w: str) -> float:
        num = self.bigrams[(w_prev, w)] + self.alpha
        den = self.context_counts[w_prev] + self.alpha * self.V
        return num / den

    def p_unigram_add_alpha(self, w: str) -> float:
        num = self.unigrams[w] + self.alpha
        den = self.total_tokens + self.alpha * self.V
        return num / den

    def p_bigram_backoff(self, w_prev: str, w: str) -> float:
        if self.bigrams[(w_prev, w)] > 0:
            return self.p_bigram_add_alpha(w_prev, w)
        else:
            return self.p_unigram_add_alpha(w)


s_lm = SmoothedBigramLM(corpus, alpha=0.5)

print("Smoothed p('transparent' | 'so') =", s_lm.p_bigram_backoff("so", "transparent"))
print("Smoothed p('translucent' | 'so') =", s_lm.p_bigram_backoff("so", "translucent"))
print("Smoothed p('unknownword' | 'so') =", s_lm.p_bigram_backoff("so", "unknownword"))

Smoothed p('transparent' | 'so') = 0.15789473684210525
Smoothed p('translucent' | 'so') = 0.15789473684210525
Smoothed p('unknownword' | 'so') = 0.012658227848101266


### 3.1 Log-Probability and Perplexity with Smoothing

We now recompute log-probabilities and perplexity using the smoothed bigram model with backoff.

In [12]:
def sentence_logprob_smoothed(lm: SmoothedBigramLM, sent_tokens: List[str]) -> float:
    logp = 0.0
    for w_prev, w_cur in zip(sent_tokens[:-1], sent_tokens[1:]):
        p = lm.p_bigram_backoff(w_prev, w_cur)
        logp += math.log(p)
    return logp


def corpus_perplexity_smoothed(lm: SmoothedBigramLM, corpus: List[List[str]]) -> float:
    total_logp = 0.0
    total_tokens = 0
    for sent in corpus:
        total_logp += sentence_logprob_smoothed(lm, sent)
        total_tokens += len(sent) - 1
    return math.exp(- total_logp / total_tokens)


print("Perplexity (smoothed) on training corpus:", corpus_perplexity_smoothed(s_lm, corpus))

# Compare on a test sentence
test_sent = prepare_corpus(["Its water is so translucent."])[0]
print("\nTest sentence:", test_sent)
print("Unsmoothed log P:", sentence_logprob(lm_bigram, test_sent))
print("Smoothed   log P:", sentence_logprob_smoothed(s_lm, test_sent))

Perplexity (smoothed) on training corpus: 4.202499870363083

Test sentence: ['<s>', 'its', 'water', 'is', 'so', 'translucent', '.', '</s>']
Unsmoothed log P: -1.3862943611198906
Smoothed   log P: -10.049756888171437


## 4. Example: Spam vs Ham Classification using Language Models

We now use **two** smoothed bigram LMs:

- One trained on *spam* emails
- One trained on *ham* (normal) emails

Given a new email, we compute:

$ \log P(\text{email} \mid \text{SPAM}) \quad \text{and} \quad \log P(\text{email} \mid \text{HAM}) $

We classify as SPAM if:

$ \log P(\text{email} \mid \text{SPAM}) > \log P(\text{email} \mid \text{HAM}). $

步骤：
1. 用语料库分别训练spamML，hamML
2. 然后获得一封新的邮件，计算概率并比较

In [13]:
spam_raw = [
    "I am foreign royalty fleeing my home country. I have wealth I need to move.",
    "Without a bank account I cannot do it. That is why I need your help.",
    "You have won a lottery prize of one million dollars. Claim your reward now.",
]

ham_raw = [
    "Hi, are we still meeting for the project discussion tomorrow?",
    "Please find attached the slides for the NLP lecture.",
    "Let us schedule a meeting next week to discuss the assignment.",
]

spam_corpus = prepare_corpus(spam_raw)
ham_corpus = prepare_corpus(ham_raw)

spam_lm = SmoothedBigramLM(spam_corpus, alpha=0.5)
ham_lm = SmoothedBigramLM(ham_corpus, alpha=0.5)

print("Spam corpus example:", spam_corpus[0])
print("Ham corpus example :", ham_corpus[0])


Spam corpus example: ['<s>', 'i', 'am', 'foreign', 'royalty', 'fleeing', 'my', 'home', 'country', '.', 'i', 'have', 'wealth', 'i', 'need', 'to', 'move', '.', '</s>']
Ham corpus example : ['<s>', 'hi', ',', 'are', 'we', 'still', 'meeting', 'for', 'the', 'project', 'discussion', 'tomorrow', '?', '</s>']


In [14]:
def classify_email(text: str) -> Tuple[str, float, float]:
    tokens = ["<s>"] + tokenize(text) + ["</s>"]
    logp_spam = sentence_logprob_smoothed(spam_lm, tokens)
    logp_ham = sentence_logprob_smoothed(ham_lm, tokens)

    if logp_spam > logp_ham:
        label = "SPAM"
    else:
        label = "HAM"
    return label, logp_spam, logp_ham


test_emails = [
    "I am foreign royalty and I need your help with my bank account.",
    "Can you send me the slides for the NLP lecture?",
    "You have won a free prize. Claim it now!",
]

for mail in test_emails:
    label, ls, lh = classify_email(mail)
    print("EMAIL:", mail)
    print(f" -> predicted: {label} (logP_spam={ls:.2f}, logP_ham={lh:.2f})")
    print("-" * 60)

EMAIL: I am foreign royalty and I need your help with my bank account.
 -> predicted: SPAM (logP_spam=-45.88, logP_ham=-66.11)
------------------------------------------------------------
EMAIL: Can you send me the slides for the NLP lecture?
 -> predicted: HAM (logP_spam=-57.16, logP_ham=-39.00)
------------------------------------------------------------
EMAIL: You have won a free prize. Claim it now!
 -> predicted: SPAM (logP_spam=-40.75, logP_ham=-51.75)
------------------------------------------------------------


## 5. Subword Tokenization with Byte Pair Encoding (BPE)

Word-level tokenization has two major issues:

1. **Out-of-vocabulary (OOV) words**
2. Very large vocabularies

Modern LMs typically use **subword tokenization**, such as BPE, WordPiece, or Unigram LM.

Here we implement a classic character-level **Byte Pair Encoding (BPE)** tokenizer and train it on a larger corpus:

- Preferably **WikiText-2** via Hugging Face `datasets`
- If unavailable, we fall back to a local file `data/corpus.txt` (you can upload any public-domain text there)

We will:

1. Load the corpus
2. Build a word frequency vocabulary
3. Train BPE: learn merge operations
4. Encode new words/sentences using the learned merges

In [15]:
def load_corpus_text(max_chars: int = 500_000) -> str:
    """Load a reasonably large open corpus as raw text.
    Priority:
    1. HuggingFace 'wikitext-2-raw-v1' if datasets is installed/available.
    2. A local UTF-8 text file at 'data/corpus.txt'.
    """
    text = ""

    try:
        from datasets import load_dataset  # type: ignore
    except ImportError:
        print("Installing 'datasets' library...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "datasets"])
        from datasets import load_dataset  # type: ignore

    try:
        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        text = "\n".join(ds["text"])
        print("Loaded WikiText-2 via HuggingFace datasets.")
    except Exception as e:
        print("Could not load WikiText-2 from datasets (", e, ")")
        local_path = "data/corpus.txt"
        if os.path.exists(local_path):
            with open(local_path, "r", encoding="utf-8") as f:
                text = f.read()
            print(f"Loaded local corpus from: {local_path}")
        else:
            raise RuntimeError(
                "No corpus found. Either ensure internet for WikiText-2, "
                "or place a text file at data/corpus.txt"
            )

    if max_chars is not None and len(text) > max_chars:
        text = text[:max_chars]
        print(f"Truncated corpus to {max_chars} characters.")
    print(f"Corpus length: {len(text):,} characters.")
    return text


corpus_text = load_corpus_text(max_chars=500_000)
corpus_text[:500]

Installing 'datasets' library...
  Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)


You should consider upgrading via the '/Users/xianzhang/venv/bin/python -m pip install --upgrade pip' command.


Loaded WikiText-2 via HuggingFace datasets.
Truncated corpus to 500000 characters.
Corpus length: 500,000 characters.


'\n = Valkyria Chronicles III = \n\n\n Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs'

### 5.1 Building a Word Vocabulary

We split the corpus on whitespace to get **word tokens** and count them. This is a crude segmentation, but enough for BPE:

- We limit ourselves to a maximum number of word tokens for speed.
- We then have a `Counter` mapping word → frequency.

In [16]:
def build_word_vocab(text: str, max_words: int = 200_000) -> Counter:
    """Build a word frequency vocabulary from raw text.
    We keep at most `max_words` tokens to keep training manageable."""
    words = text.split()
    if len(words) > max_words:
        words = words[:max_words]
        print(f"Using first {max_words:,} words for BPE training.")
    vocab = Counter(words)
    print(f"Unique words in vocab: {len(vocab):,}")
    return vocab


word_vocab = build_word_vocab(corpus_text, max_words=200_000)
list(word_vocab.items())[:20]

Unique words in vocab: 12,365


[('=', 1291),
 ('Valkyria', 54),
 ('Chronicles', 39),
 ('III', 17),
 ('Senjō', 5),
 ('no', 57),
 ('3', 52),
 (':', 130),
 ('Unrecorded', 1),
 ('(', 486),
 ('Japanese', 7),
 ('戦場のヴァルキュリア3', 3),
 (',', 4782),
 ('lit', 5),
 ('.', 3397),
 ('of', 2731),
 ('the', 5248),
 ('Battlefield', 4),
 (')', 487),
 ('commonly', 5)]

### 5.2 BPE Training: Byte Pair Encoding

Classic BPE algorithm (character-level version):

1. Represent each word as a sequence of **characters** plus an end-of-word marker `</w>`.
   - Example: `"water"` → `['w', 'a', 't', 'e', 'r', '</w>']`
2. Count all adjacent symbol pairs across the vocabulary (weighted by word frequency).
3. Find the most frequent pair `(A, B)`.
4. Merge it into a new symbol `AB` and update the vocabulary.
5. Repeat for `N` merges.

We implement:

- `train_bpe()` → returns the list of merges in order.
- Helpers for initializing symbol vocabulary, counting pairs, and applying a merge.

In [17]:
def init_symbol_vocab(word_vocab: Counter) -> dict:
    """Represent words as sequences of characters + </w>.
    Returns a dict: { 'w a t e r </w>': frequency }"""
    symbol_vocab = {}  # type: ignore
    for word, freq in word_vocab.items():
        chars = list(word)
        token = " ".join(chars + ["</w>"])
        symbol_vocab[token] = freq
    return symbol_vocab


def get_pair_stats(symbol_vocab: dict) -> Counter:
    """Count frequency of each adjacent symbol pair across the vocab.
    symbol_vocab: { 'w a t e r </w>': freq }"""
    pairs = Counter()
    for token, freq in symbol_vocab.items():
        symbols = token.split()
        for pair in zip(symbols, symbols[1:]):
            """ symbols = ["w", "a", "t", "e", "r", "</w>"]
            zip(symbols, symbols[1:]) 生成：
            ("w", "a")
            ("a", "t")
            ("t", "e")
            ("e", "r")
            ("r", "</w>")
            """
            pairs[pair] += freq
    return pairs
"""
Counter({
    ("w", "a"): 35,
    ("a", "t"): 28,
    ("t", "e"): 20,
    ("e", "r"): 20,
    ("r", "</w>"): 20
})
"""

# "t h e </w>" → "t he </w>" 合并he，如果he是最频繁的pair
def merge_pair(symbol_vocab: dict, pair: tuple) -> dict:
    """Merge all occurrences of pair (A, B) into new symbol 'AB' in symbol_vocab."""
    A, B = pair
    pattern = re.escape(A + " " + B)
    merged_symbol = A + B
    new_symbol_vocab = {}
    for token, freq in symbol_vocab.items():
        new_token = re.sub(r"(?<!\S)" + pattern + r"(?!\S)", merged_symbol, token)
        new_symbol_vocab[new_token] = freq
    return new_symbol_vocab
"""输入symbol_vocab
"t h e </w>" : 10
"h e y </w>": 5
"w h e n </w>": 12
输出symbol_vocab
"t he </w>" : 10
"he y </w>" : 5
"w he n </w>" : 12
注意，之前的word_vocab是：
[
('the':10),
('hey': 5),
('when':12)
]
"""


def train_bpe(word_vocab: Counter, num_merges: int = 300) -> list:
    """Train BPE on a word vocab.
    Returns a list of merges: [(A,B), (C,D), ...] in merge order."""
    symbol_vocab = init_symbol_vocab(word_vocab)
    merges = []

    for i in range(num_merges):
        pair_stats = get_pair_stats(symbol_vocab)
        if not pair_stats:
            print("No more pairs to merge.")
            break

        (best_pair, best_freq) = pair_stats.most_common(1)[0]
        if best_freq < 2:
            print(f"Stopping early at step {i}: max pair frequency < 2")
            break

        merges.append(best_pair)
        symbol_vocab = merge_pair(symbol_vocab, best_pair)

        if (i + 1) % 50 == 0:
            print(f"Merge step {i+1}: merged {best_pair} (freq={best_freq})")

    print(f"Total merges learned: {len(merges)}")
    return merges


bpe_merges = train_bpe(word_vocab, num_merges=1000)
bpe_merges[:20]

Merge step 50: merged ('a', 't</w>') (freq=1181)
Merge step 100: merged ('b', 'y</w>') (freq=549)
Merge step 150: merged ('th', 'er</w>') (freq=341)
Merge step 200: merged ('s', 'h') (freq=238)
Merge step 250: merged ('b', 'er</w>') (freq=184)
Merge step 300: merged ('as', 's') (freq=153)
Merge step 350: merged ('mor', 'e</w>') (freq=125)
Merge step 400: merged ('i', 'z') (freq=108)
Merge step 450: merged ('k', 's</w>') (freq=96)
Merge step 500: merged ('m', 'us') (freq=85)
Merge step 550: merged ('op', 'er') (freq=77)
Merge step 600: merged ('re', 'pres') (freq=70)
Merge step 650: merged ('ri', 'a</w>') (freq=63)
Merge step 700: merged ('b', 'ers</w>') (freq=58)
Merge step 750: merged ('cont', 'in') (freq=54)
Merge step 800: merged ('s', 'ame</w>') (freq=50)
Merge step 850: merged ('C', 'ull') (freq=47)
Merge step 900: merged ('C', 'on') (freq=44)
Merge step 950: merged ('li', 'gh') (freq=42)
Merge step 1000: merged ('g', 'overn') (freq=40)
Total merges learned: 1000


[('e', '</w>'),
 ('s', '</w>'),
 ('t', 'h'),
 ('d', '</w>'),
 ('n', '</w>'),
 ('e', 'r'),
 ('t', '</w>'),
 ('th', 'e</w>'),
 ('i', 'n'),
 ('a', 'n'),
 (',', '</w>'),
 ('y', '</w>'),
 ('e', 'd</w>'),
 ('o', 'r'),
 ('.', '</w>'),
 ('a', 'r'),
 ('a', 'l'),
 ('t', 'i'),
 ('r', 'e'),
 ('o', '</w>')]

### 5.3 BPE Encoding for New Words

We now turn the merge list into a **rank dictionary** and define functions:

- `bpe_encode_word(word, merge_ranks)`
- `bpe_encode_text(text, merge_ranks)`

Encoding algorithm:

1. Start with characters + `</w>`.
2. Repeatedly merge the pair with the **lowest rank** (highest priority) present in the sequence.
3. Stop when no pair in the sequence appears in the merge list.
4. Remove the final `</w>` marker.

Result: a sequence of subword tokens for each word.

In [18]:
def build_merge_ranks(merges: list) -> dict:
    """Turn merge list into a dict mapping pair -> rank (lower rank = higher priority)."""
    return {pair: i for i, pair in enumerate(merges)}


bpe_ranks = build_merge_ranks(bpe_merges)


def bpe_encode_word(word: str, merge_ranks: dict) -> List[str]:
    """Encode a single word using BPE merges.
    Returns a list of subword tokens (without the </w> marker)."""
    tokens = list(word) + ["</w>"]

    while True:
        # 组出所有的可合并的字符对
        pairs = [(tokens[i], tokens[i + 1]) for i in range(len(tokens) - 1)]
        candidate = None
        candidate_rank = None

        # 循环遍历更新，找出优先级最高的pair
        # Q：但是这样不是一直都合并的是同一个pair吗？就是优先级最高的那个？
        # A: 每次合并之后，tokens 变了 → pairs 也变了 → 下一轮要重新找“rank 最小的 pair”
        for pair in pairs:
            if pair in merge_ranks:
                rank = merge_ranks[pair]
                if candidate_rank is None or rank < candidate_rank:
                    candidate_rank = rank
                    candidate = pair

        if candidate is None:
            break  # no more merges possible

        new_tokens = []
        i = 0
        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == candidate[0]
                and tokens[i + 1] == candidate[1]
            ): # 如果找到目标candidate pair就拼起来
                new_tokens.append(tokens[i] + tokens[i + 1])
                i += 2
            else: # 不是目标，就正常过
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    if tokens and tokens[-1] == "</w>": #移除“</w>”
        tokens = tokens[:-1]
    tokens = [t.replace("</w>", "") for t in tokens]
    return tokens


def bpe_encode_text(text: str, merge_ranks: dict) -> List[str]:
    """Encode a whitespace-separated text string into a list of subword tokens."""
    tokens: List[str] = []
    for word in text.split():
        tokens.extend(bpe_encode_word(word, merge_ranks))
    return tokens


# Quick demo
for w in ["water", "transparent", "translucent", "students", "royalty", "million"]:
    print(w, "->", bpe_encode_word(w.lower(), bpe_ranks))

water -> ['w', 'ater']
transparent -> ['tran', 'sp', 'ar', 'ent']
translucent -> ['tran', 's', 'l', 'u', 'c', 'ent']
students -> ['st', 'ud', 'ents']
royalty -> ['ro', 'y', 'al', 'ty']
million -> ['mil', 'lion']


### 5.4 BPE Vocabulary Size and Example Encodings

We now approximate the **subword vocabulary** by encoding a subset of words, and compare:

- Original word vocabulary size
- Approximate subword vocabulary size

We also apply the BPE tokenizer to the earlier **spam/ham** example sentences to see the subword segmentation in action.

从已有的 word-level 词表（word_vocab）中，使用 BPE 对前 max_words 个高频词进行编码，从而估计一个子词词表。

- word_vocab：一个 Counter，统计每个单词出现次数
- ge_ranks：BPE 的 merge 表（pair → rank）
- max_words：只处理前 10k 或 20k 个词，减少计算量

如果 word_vocab 是：
```
{"lower": 123, "slowest": 87, ...}
```

BPE会生成subword：
```
lower → low + er
slowest → slow + est
```
最终 subword_vocab 类似：
```
{
  "low": 100,
  "er": 150,
  "slow": 80,
  "est": 87,
  ...
}
```


In [19]:

def derive_subword_vocab(word_vocab: Counter, merge_ranks: dict, max_words: int = 10_000) -> Counter:
    """Approximate subword vocab by encoding a subset of words."""
    subword_vocab = Counter()
    for i, word in enumerate(word_vocab.keys()):
        if i >= max_words:
            break
        pieces = bpe_encode_word(word, merge_ranks)
        for p in pieces:
            subword_vocab[p] += 1
    return subword_vocab


subword_vocab = derive_subword_vocab(word_vocab, bpe_ranks, max_words=20_000)

print(f"Original word vocab size: {len(word_vocab):,}")
print(f"Approx. subword vocab size (from 20k words): {len(subword_vocab):,}")

print("\nSample subword tokens:", list(subword_vocab.keys())[:50])

example_sentences = [
    "I am foreign royalty and I need your help with my bank account.",
    "You have won a lottery prize of one million dollars. Claim your reward now!",
    "Please find attached the slides for the NLP lecture.",
]

for s in example_sentences:
    print("TEXT:", s)
    print("TOKENS:", bpe_encode_text(s.lower(), bpe_ranks))
    print("-" * 80)

Original word vocab size: 12,365
Approx. subword vocab size (from 20k words): 1,018

Sample subword tokens: ['=', 'Valkyria', 'Chronic', 'les', 'I', 'II', 'S', 'en', 'j', 'ō', 'no', '3', ':', 'Un', 'recor', 'ded', '(', 'J', 'ap', 'an', 'ese', '戦', '場', 'の', 'ヴ', 'ァ', 'ル', 'キ', 'ュ', 'リ', 'ア', ',', 'li', 't', '.', 'of', 'the', 'B', 'att', 'le', 'fi', 'el', 'd', ')', 'comm', 'only', 're', 'fer', 'red', 'to']
TEXT: I am foreign royalty and I need your help with my bank account.
TOKENS: ['i', 'a', 'm', 'for', 'e', 'ig', 'n', 'ro', 'y', 'al', 'ty', 'and', 'i', 'n', 'e', 'ed', 'y', 'our', 'hel', 'p', 'with', 'my', 'b', 'an', 'k', 'acc', 'oun', 't', '.']
--------------------------------------------------------------------------------
TEXT: You have won a lottery prize of one million dollars. Claim your reward now!
TOKENS: ['y', 'ou', 'have', 'w', 'on', 'a', 'l', 'ot', 'ter', 'y', 'pri', 'z', 'e', 'of', 'one', 'mil', 'lion', 'doll', 'ar', 's', '.', 'cl', 'a', 'i', 'm', 'y', 'our', 're', 'w', 'a

## 6. Exercises

1. **Tokenization variants**  
   - Modify `tokenize()` to keep case (no lowercasing).  
   - Compare vocabulary size and perplexity of your language model with and without lowercasing.

    Answer:
    V_no_lowercase  >  V_lowercase

    ```
    Vocabulary Size 会显著变大

    使用 lowercasing：
    `"The", "the", "THE"` → 都变成 `"the"`
    所以这些 token 是相同的。

    不使用 lowercasing：
    `"The"、"the"、"THE"`是不同 token
    会被当作不同的词加入 vocab → vocab size 增加。

    大小写不同的词频会被视为不同 token，导致：1. 出现更多 unique tokens 2.词表变大 3.更多低频词（稀有词）
    ```

    Perplexity（困惑度）会变高

    ```
    1. 稀有词变多 → 概率变低
    2. More Out-of-vocabulary (OOV) events
    3. 大写分布更难建模
    ```


2. **Trigram model**  
   Extend `NGramLM` to support `n=3`.  
   Implement trigram counts and probabilities $  p(w_i \mid w_{i-2}, w_{i-1}) $
   Compare perplexity with bigram and unigram models.

3. **Smoothing comparison**  
   Implement pure add-α bigram smoothing (no backoff).  
   Compare perplexity and spam/ham classification accuracy for different α values.
   ```
   1. α 很小（接近 0，例如 0.01）
   模型几乎变回 MLE：
   - 见过的 bigram 概率很高
   - 没见过的 bigram 概率 ≈ α / (c + αV) 很小
   test 里 unseen bigram 多 → 句子概率非常小 → perplexity 偏高

   2. α 适中（0.1 ~ 1 左右）
   - 未见 bigram 得到合理的小概率
   - 见过 bigram 的概率稍被拉低，但仍体现频率差异
   - overfitting 减少，总体概率分布更合理 ➜ perplexity 最低（最佳点）

   3. α 很大（比如 5, 10）
   分子 ≈ α，分母 ≈ αV，所有 bigram 概率都接近 1/V
   模型几乎不区分高频/低频 bigram，等价于“接近均匀分布” ➜ 语言信息被抹掉，perplexity 又升高
   ```

4. **Subword-level LM**  
   - BPE-encode your training sentences (spam + ham).  
   - Train a unigram/bigram LM over subword tokens instead of word tokens.  
   - Compare perplexity and robustness to OOV words.

In [20]:
def prepare_corpus_bpe(raw_sentences: List[str]) -> List[List[str]]:
    corpus = []
    for s in raw_sentences:
        # toks = tokenize(s)
        toks = bpe_encode_text(s.lower(), bpe_ranks)
        if not toks:
            continue
        corpus.append(["<s>"] + toks + ["</s>"])
    return corpus

spam_corpus_bpe = prepare_corpus_bpe(spam_raw)
ham_corpus_bpe = prepare_corpus_bpe(ham_raw)

spam_lm_bpe = SmoothedBigramLM(spam_corpus_bpe, alpha=0.5)
ham_lm_bpe = SmoothedBigramLM(ham_corpus_bpe, alpha=0.5)

def classify_email_bpe(text: str) -> Tuple[str, float, float]:
    tokens = ["<s>"] + bpe_encode_text(text.lower(), bpe_ranks) + ["</s>"]
    logp_spam = sentence_logprob_smoothed(spam_lm, tokens)
    logp_ham = sentence_logprob_smoothed(ham_lm, tokens)

    if logp_spam > logp_ham:
        label = "SPAM"
    else:
        label = "HAM"
    return label, logp_spam, logp_ham

print("Train perplexity on spam_raw")
print("word tokens: ", corpus_perplexity_smoothed(spam_lm, spam_corpus))
print("subword tokens: ", corpus_perplexity_smoothed(spam_lm_bpe, spam_corpus_bpe))

for mail in test_emails:
    label, ls, lh = classify_email(mail)
    label_bpe, ls_bpe, lh_bpe = classify_email_bpe(mail)
    print("EMAIL:", mail)
    print(f" -> predicted: {label} (logP_spam={ls:.2f}, logP_ham={lh:.2f}), spam/ham: {ls/lh:.2f}")
    print(f" -> bpe_predicted: {label_bpe} (logP_spam={ls_bpe:.2f}, logP_ham={lh_bpe:.2f}), spam/ham: {ls_bpe/lh_bpe:.2f}")
    print("-" * 60)


Train perplexity on spam_raw
word tokens:  13.48671751879896
subword tokens:  21.523612178154103
EMAIL: I am foreign royalty and I need your help with my bank account.
 -> predicted: SPAM (logP_spam=-45.88, logP_ham=-66.11), spam/ham: 0.69
 -> bpe_predicted: HAM (logP_spam=-137.51, logP_ham=-133.91), spam/ham: 1.03
------------------------------------------------------------
EMAIL: Can you send me the slides for the NLP lecture?
 -> predicted: HAM (logP_spam=-57.16, logP_ham=-39.00), spam/ham: 1.47
 -> bpe_predicted: HAM (logP_spam=-103.42, logP_ham=-88.69), spam/ham: 1.17
------------------------------------------------------------
EMAIL: You have won a free prize. Claim it now!
 -> predicted: SPAM (logP_spam=-40.75, logP_ham=-51.75), spam/ham: 0.79
 -> bpe_predicted: SPAM (logP_spam=-92.53, logP_ham=-92.96), spam/ham: 1.00
------------------------------------------------------------



5. **Compare with production tokenizers**
   - Use a pretrained Hugging Face tokenizer (e.g., GPT-2 or RoBERTa).
   - Compare their subword segmentation with your BPE implementation on the same sentences.
   - Discuss differences (byte-level vs char-level, special tokens, etc.).

In [21]:
import sys
print(sys.executable)
!{sys.executable} -m pip install "protobuf<5" "transformers<4.46"



/Users/xianzhang/venv/bin/python
You should consider upgrading via the '/Users/xianzhang/venv/bin/python -m pip install --upgrade pip' command.


In [23]:
from transformers import AutoTokenizer

hf_tok = AutoTokenizer.from_pretrained(
    "gpt2",
    use_fast=False,
)

for s in test_emails:
    print("=== Sentence ===")
    print(s)

    print("\nYour BPE:")
    print(bpe_encode_text(s.lower(), bpe_ranks))

    print("\nGPT2 tokenizer BPE:")
    print(hf_tok.tokenize(s))

    print("-" * 60)




=== Sentence ===
I am foreign royalty and I need your help with my bank account.

Your BPE:
['i', 'a', 'm', 'for', 'e', 'ig', 'n', 'ro', 'y', 'al', 'ty', 'and', 'i', 'n', 'e', 'ed', 'y', 'our', 'hel', 'p', 'with', 'my', 'b', 'an', 'k', 'acc', 'oun', 't', '.']

GPT2 tokenizer BPE:
['I', 'Ġam', 'Ġforeign', 'Ġroyalty', 'Ġand', 'ĠI', 'Ġneed', 'Ġyour', 'Ġhelp', 'Ġwith', 'Ġmy', 'Ġbank', 'Ġaccount', '.']
------------------------------------------------------------
=== Sentence ===
Can you send me the slides for the NLP lecture?

Your BPE:
['can', 'y', 'ou', 'send', 'me', 'the', 's', 'li', 'd', 'es', 'for', 'the', 'n', 'l', 'p', 'l', 'ec', 'tu', 're', '?']

GPT2 tokenizer BPE:
['Can', 'Ġyou', 'Ġsend', 'Ġme', 'Ġthe', 'Ġslides', 'Ġfor', 'Ġthe', 'ĠN', 'LP', 'Ġlecture', '?']
------------------------------------------------------------
=== Sentence ===
You have won a free prize. Claim it now!

Your BPE:
['y', 'ou', 'have', 'w', 'on', 'a', 'f', 'ree', 'pri', 'z', 'e', '.', 'cl', 'a', 'i', 'm', 'it',